# 05b — Corrected Table 7 cross-cell retention

Corrected cross-cell RQ2/Table 7 analysis.

The historical implementation in `05_analysis_and_figures.ipynb` is preserved unchanged.

### Analysis

- Primary cohort: RULER NIAH, n=60
- Cells: GatedDeltaNet, DeltaNet, Mamba2
- Source: corrected Run 026 `04i_ruler_controls_rows.json`
- Layer: 27
- Condition: ordered
- Readout: J-Lens
- Only actually evicted needles

The original Table 7 used pairwise ratios of exponential `p_mem` half-lives.
Corrected Run 026 uses digit-sequence rank/log-probability instead.

Therefore the original ≥2× half-life comparison is reported as **not testable**.
No post-hoc replacement inferential procedure is introduced.


In [1]:
import sys
from pathlib import Path

# Resolve repo root whether executed from repo/ or repo/notebooks/.
ROOT = Path.cwd()
if not (ROOT / "ahn_interp.py").exists():
    ROOT = ROOT.parent

assert (ROOT / "ahn_interp.py").exists(), f"Repo root not found from {Path.cwd()}"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ahn_interp as ai
from ruler_retention_curve import curve_for_layer

LAYER = 27

PATHS = {
    "GatedDeltaNet": ROOT / "results/run_3b_gdn/04i_ruler_controls_rows.json",
    "DeltaNet": ROOT / "results/run_3b_dn/04i_ruler_controls_rows.json",
    "Mamba2": ROOT / "results/run_3b_m2/04i_ruler_controls_rows.json",
}

OUTPUT = ROOT / "results/05_table7_crosscell_retention_corrected.json"

MATCH_FIELDS = (
    "example",
    "n_tokens",
    "needle_pos",
    "compression_boundary",
    "eviction_distance",
    "needle_is_evicted",
    "ruler_config",
)

for cell_name, source_path in PATHS.items():
    assert source_path.exists(), f"{cell_name}: missing {source_path}"

print("repo root:", ROOT)
print("All source files found.")


repo root: /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks
All source files found.


In [2]:
def analysis_rows(path):
    """Load the corrected primary RULER rows used by Table 7."""
    blob = ai.load_json(path.name, str(path.parent))

    return [
        r for r in blob["rows"]
        if r["layer"] == LAYER
        and r["condition"] == "ordered"
        and r["readout"] == "jlens"
        and r["needle_is_evicted"]
    ]


def row_signature(row):
    """Architecture-independent cohort/placement identity."""
    return tuple(row[field] for field in MATCH_FIELDS)


In [3]:
rows_by_cell = {
    cell: analysis_rows(path)
    for cell, path in PATHS.items()
}

for cell, rows in rows_by_cell.items():
    distances = [r["eviction_distance"] for r in rows]

    print(
        f"{cell:15s}",
        f"n={len(rows):2d}",
        f"distance={min(distances)}..{max(distances)}"
    )


GatedDeltaNet   n=32 distance=46..7414
DeltaNet        n=32 distance=46..7414
Mamba2          n=32 distance=46..7414


In [4]:
# Verify that cross-cell comparisons use exactly the same examples/placements.

reference_name = "GatedDeltaNet"
reference = [
    row_signature(r)
    for r in rows_by_cell[reference_name]
]

for cell, rows in rows_by_cell.items():
    current = [row_signature(r) for r in rows]

    assert current == reference, (
        f"{cell}: cohort/placement metadata does not match {reference_name}"
    )

print(
    f"Matched cohort validated: "
    f"{len(reference)} identical evicted L{LAYER} examples per cell."
)


Matched cohort validated: 32 identical evicted L27 examples per cell.


In [5]:
# Reuse the corrected retention analysis already implemented in
# ruler_retention_curve.py.

results = {
    cell: curve_for_layer(rows)
    for cell, rows in rows_by_cell.items()
}

for cell, res in results.items():
    print(f"\n{cell}")
    print(f"  n              : {res['n']}")
    print(f"  distance range : {res['eviction_distance_range']}")
    print(f"  median rank    : {res['overall_median_rank']:.1f}")
    print(f"  Spearman rho   : {res['spearman_rho']:+.4f}")
    print(f"  permutation p  : {res['perm_p']:.4f}")
    print(f"  verdict        : {res['verdict']}")



GatedDeltaNet
  n              : 32
  distance range : [46.0, 7414.0]
  median rank    : 49775.9
  Spearman rho   : +0.0656
  permutation p  : 0.7240
  verdict        : flat -- no distance dependence over the range measured

DeltaNet
  n              : 32
  distance range : [46.0, 7414.0]
  median rank    : 38581.4
  Spearman rho   : +0.0249
  permutation p  : 0.8896
  verdict        : flat -- no distance dependence over the range measured

Mamba2
  n              : 32
  distance range : [46.0, 7414.0]
  median rank    : 76951.2
  Spearman rho   : +0.0993
  permutation p  : 0.5846
  verdict        : flat -- no distance dependence over the range measured


In [6]:
# Inspect the measured curves rather than relying only on the aggregate test.

for cell, res in results.items():
    print(f"\n{cell}")

    for b in res["bins"]:
        print(
            f"  distance ~{b['mean_eviction_distance']:6.0f}"
            f" | n={b['n']:2d}"
            f" | median rank={b['median_rank']:8.0f}"
            f" | beat chance={b['frac_beating_chance']:.2f}"
        )



GatedDeltaNet
  distance ~  1850 | n= 8 | median rank=   47241 | beat chance=1.00
  distance ~  3441 | n= 8 | median rank=   61841 | beat chance=0.88
  distance ~  5058 | n= 8 | median rank=   42040 | beat chance=1.00
  distance ~  6463 | n= 8 | median rank=   60196 | beat chance=0.62

DeltaNet
  distance ~  1850 | n= 8 | median rank=   44866 | beat chance=1.00
  distance ~  3441 | n= 8 | median rank=   31029 | beat chance=1.00
  distance ~  5058 | n= 8 | median rank=   38581 | beat chance=1.00
  distance ~  6463 | n= 8 | median rank=   37882 | beat chance=0.88

Mamba2
  distance ~  1850 | n= 8 | median rank=   73117 | beat chance=0.62
  distance ~  3441 | n= 8 | median rank=   74111 | beat chance=0.50
  distance ~  5058 | n= 8 | median rank=   76951 | beat chance=0.38
  distance ~  6463 | n= 8 | median rank=   94692 | beat chance=0.38


## Interpretation

Table 7's original pre-registered analysis requires a defensible retention
half-life that can be compared across GatedDeltaNet, DeltaNet, and Mamba2.

The validated primary RQ2 cohort is now RULER NIAH n=60. The earlier homemade
controlled distance sweep is excluded from the primary Table 7 analysis because
that small synthetic construction repeatedly failed to generalize.

The RULER cohort provides naturally varying, example-specific eviction distances
rather than a controlled repeated distance sweep of the same targets. The
corrected Run 026 L27 rank-vs-distance analysis is therefore retained as a
descriptive robustness analysis, not treated as a replacement definition of
the pre-registered `P_mem` half-life.

Accordingly:

- homemade retention half-lives are not used
- no cross-construction comparison is used
- exponential half-life ratios are not estimated
- the pre-registered >=2x half-life criterion is not testable
- bootstrap ratio CIs are not computed
- Holm adjustment is not applicable
- no post-hoc replacement inferential test is invented

This is reported as **not estimable under the validated analysis design**, not
as failure of the >=2x effect-size criterion.


In [7]:
distance_ranges = {
    tuple(res["eviction_distance_range"])
    for res in results.values()
}
assert len(distance_ranges) == 1, "Cross-cell distance ranges differ."

summary = {
    "table": 7,
    "status": "preregistered_half_life_comparison_not_estimable",
    "analysis_type": "corrected_cross_cell_ruler_retention",
    "primary_cohort": "RULER NIAH n=60",
    "analysis_layer": LAYER,

    "conditions": {
        "condition": "ordered",
        "readout": "jlens",
        "needle_is_evicted": True,
    },

    "source_files": {
        cell: str(path)
        for cell, path in PATHS.items()
    },

    "eviction_distance_range": list(next(iter(distance_ranges))),

    "cells": {
        cell: {
            "n_evicted": res["n"],
            "median_rank": res["overall_median_rank"],
            "spearman_rho": res["spearman_rho"],
            "permutation_p": res["perm_p"],
            "distance_verdict": res["verdict"],
        }
        for cell, res in results.items()
    },

    "ruler_distance_analysis": {
        "role": "descriptive_only",
        "reason": (
            "RULER provides naturally varying example-specific eviction distances, "
            "not a controlled repeated distance sweep of the same targets. The "
            "rank-vs-distance result is therefore not used to redefine the "
            "pre-registered retention half-life."
        ),
    },

    "preregistered_half_life_test": {
        "estimable": False,
        "two_x_ratio_criterion_testable": False,
        "bootstrap_ratio_ci": None,
        "holm_adjustment": None,
        "reason": (
            "The validated primary RQ2 cohort is RULER NIAH n=60. The earlier "
            "homemade controlled distance sweep is excluded from primary inference "
            "because that synthetic construction repeatedly failed to generalize. "
            "Under the validated RULER design there is no defensible controlled "
            "half-life estimand for the original cross-cell Table 7 comparison."
        ),
    },

    "homemade_distance_sweep": {
        "used_for_primary_table7": False,
        "role": "supporting_or_limitations_only",
    },

    "fallback_inference": {
        "performed": False,
        "reason": (
            "No pre-registered cross-cell inferential procedure exists for a "
            "replacement nonparametric distance statistic, so no post-hoc test "
            "is introduced."
        ),
    },
}

ai.save_json(
    summary,
    OUTPUT.name,
    results_dir=str(OUTPUT.parent),
)

print(f"saved -> {OUTPUT}")


saved -> /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks/results/05_table7_crosscell_retention_corrected.json
